# Training Crews

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) CrewAI roadmap.

You will use the interactive **training** loop to refine agent outputs with human feedback and persist learned suggestions.

## 1. Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Run `crew.train()` (interactive)

`crew.train()` runs the same task flow as a normal kickoff, but **pauses for your feedback** after agent outputs. You type corrections or approval; the agent revises; the process repeats for **`n_iterations`**.

**Run the next cell in a terminal-friendly environment** (local Jupyter, VS Code, or similar). In read-only or non-interactive runners, stdin may not be available and the loop will not work as intended.

In [ ]:
from crewai import Agent, Crew, Process, Task

researcher = Agent(
    role="Research Analyst",
    goal="Produce accurate, well-structured notes on the topic",
    backstory="You summarize sources clearly and flag uncertainty.",
    verbose=True,
)

research_task = Task(
    description="Research {topic}. Return a short bullet list of key points.",
    expected_output="Bullet list of key points.",
    agent=researcher,
)

crew = Crew(
    agents=[researcher],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True,
)

crew.train(
    n_iterations=3,
    inputs={"topic": "AI Agents"},
    filename="trained_agents.pkl",
)

## 4. Training artifacts and persistence

The **`filename`** you pass to `train()` is where CrewAI writes the training bundle. That pickle typically holds **per-agent suggestions**, **quality scores**, and **summaries** derived from your feedback across iterations.

On later **`kickoff()`** runs, agents that load this data **append those suggestions to task prompts** automatically, so you keep the learned behavior without copying feedback into your source code. Store the file with your project or in shared storage if teams should share the same tuned crew.

## Key Takeaways

- **`crew.train()`** is an **interactive** loop: execute → review output → human feedback → revise → repeat for **`n_iterations`**.  
- **`inputs`** matches **`kickoff(inputs=...)`** so `{topic}` and other placeholders resolve during training.  
- The saved **`.pkl`** persists suggestions and scores; **future runs** apply them by **injecting** that guidance into tasks.  
- For non-interactive benchmarking, use the CLI **`crewai test`** (see the roadmap MDX) instead of `train()`.